# OmniVoice — thử Auto Voice tiếng Việt
Chọn **Runtime → Change runtime type → T4 GPU**, rồi **Run all**.
Sửa `TEXTS` ở cell 2 để thử câu khác. Audio WAV được lưu tại `/content/omnivoice_test` và nghe ngay bên dưới.

Auto Voice tự sinh giọng, không cần bản thu; giọng có thể thay đổi giữa các câu. Dùng một phiên Colab riêng cho notebook này. Phần đọc đường dẫn dataset sẽ thêm sau.
[Nguồn OmniVoice](https://github.com/k2-fsa/OmniVoice)

In [ ]:
# 1. Cài thư viện vào Python đang chạy notebook.
%pip install -q "omnivoice==0.2.1" soundfile

In [ ]:
# 2. Nhập câu muốn đọc.
TEXTS = [
    "Xin chào, đây là giọng đọc tự động bằng tiếng Việt.",
    "Nhắn cho mẹ là tối nay con về muộn mười phút.",
]
SEED = 2026

In [ ]:
# 3. Tải model lên GPU T4, FP16. Auto Voice không cần model ASR.
import torch
import soundfile as sf
from omnivoice import OmniVoice
from transformers import set_seed

assert torch.cuda.is_available(), "Hãy chọn T4 GPU trong Runtime → Change runtime type."
model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16, load_asr=False,
)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 4. Tạo WAV tiếng Việt và nghe thử.
from pathlib import Path
from IPython.display import Audio, display

output_dir = Path("/content/omnivoice_test")
output_dir.mkdir(parents=True, exist_ok=True)
set_seed(SEED)
for i, text in enumerate(TEXTS, 1):
    with torch.inference_mode():
        audios = model.generate(text=text, language="vi", num_step=32, speed=1.0)
    path = output_dir / f"test_{i:02d}.wav"
    sf.write(str(path), audios[0], model.sampling_rate, subtype="PCM_16")
    print(text, "→", path)
    display(Audio(filename=str(path)))